**Pelatihan Model Ulasan Review Pengguna Spotify Dari Playstore**

In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

**pre processing data model**

In [3]:
# Load dataset
df = pd.read_csv('spotify_reviews_200k.csv')

# Data preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df['cleaned_review'] = df['review'].astype(str).apply(clean_text)

# Labeling berdasarkan rating
def categorize_sentiment(score):
    if score >= 4:
        return 'positive'
    elif score == 3:
        return 'neutral'
    else:
        return 'negative'

df['sentiment'] = df['rating'].apply(categorize_sentiment)

# Encode labels
sentiment_labels = {'positive': 2, 'neutral': 1, 'negative': 0}
df['label'] = df['sentiment'].map(sentiment_labels)

# Tokenisasi dan Padding
max_words = 20000
max_len = 100
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['cleaned_review'])
X = tokenizer.texts_to_sequences(df['cleaned_review'])
X = pad_sequences(X, maxlen=max_len)
y = df['label'].values



**Split data dan Melatih Model LTSM**

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model LSTM
lstm_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Training LSTM
history = lstm_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=128)

# Evaluasi
y_pred = np.argmax(lstm_model.predict(X_test), axis=1)
accuracy = accuracy_score(y_test, y_pred)
print("LSTM Accuracy:", accuracy)
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

# Simpan model
lstm_model.save("lstm_sentiment.h5")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 326s 257ms/step - accuracy: 0.8586 - loss: 0.4050 - val_accuracy: 0.9704 - val_loss: 0.1037
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 341s 273ms/step - accuracy: 0.9730 - loss: 0.1088 - val_accuracy: 0.9804 - val_loss: 0.0762
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 378s 270ms/step - accuracy: 0.9798 - loss: 0.0794 - val_accuracy: 0.9830 - val_loss: 0.0632
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 366s 257ms/step - accuracy: 0.9834 - loss: 0.0657 - val_accuracy: 0.9866 - val_loss: 0.0516
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 337s 269ms/step - accuracy: 0.9853 - loss: 0.0597 - val_accuracy: 0.9851 - val_loss: 0.0604
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 33s 26ms/step


LSTM Accuracy: 0.985075
              precision    recall  f1-score   support

    negative       0.99      0.97      0.98     10071
     neutral       0.98      0.87      0.92      1946
    positive       0.98      1.00      0.99     27983

    accuracy                           0.99     40000
   macro avg       0.98      0.95      0.96     40000
weighted avg       0.99      0.99      0.98     40000



**Melatih Model CNN**

In [5]:
# Model CNN
cnn_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Conv1D(64, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

cnn_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Training CNN
history_cnn = cnn_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=128)

# Evaluasi CNN
y_pred_cnn = np.argmax(cnn_model.predict(X_test), axis=1)
accuracy_cnn = accuracy_score(y_test, y_pred_cnn)
print("CNN Accuracy:", accuracy_cnn)
print(classification_report(y_test, y_pred_cnn, target_names=['negative', 'neutral', 'positive']))

# Simpan model CNN
cnn_model.save("cnn_sentiment.h5")



Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 134s 106ms/step - accuracy: 0.8513 - loss: 0.4209 - val_accuracy: 0.9855 - val_loss: 0.0622
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 135s 100ms/step - accuracy: 0.9785 - loss: 0.0812 - val_accuracy: 0.9891 - val_loss: 0.0473
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 139s 98ms/step - accuracy: 0.9821 - loss: 0.0644 - val_accuracy: 0.9893 - val_loss: 0.0427
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 162s 114ms/step - accuracy: 0.9850 - loss: 0.0562 - val_accuracy: 0.9898 - val_loss: 0.0409
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 193s 107ms/step - accuracy: 0.9849 - loss: 0.0560 - val_accuracy: 0.9896 - val_loss: 0.0407
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step


CNN Accuracy: 0.9896
              precision    recall  f1-score   support

    negative       0.99      0.98      0.99     10071
     neutral       0.99      0.92      0.96      1946
    positive       0.99      1.00      0.99     27983

    accuracy                           0.99     40000
   macro avg       0.99      0.97      0.98     40000
weighted avg       0.99      0.99      0.99     40000



**Melatih Model GRU**

In [6]:
# Model GRU
gru_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    GRU(64, return_sequences=True),
    GRU(32),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Training GRU
history_gru = gru_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=128)

# Evaluasi GRU
y_pred_gru = np.argmax(gru_model.predict(X_test), axis=1)
accuracy_gru = accuracy_score(y_test, y_pred_gru)
print("GRU Accuracy:", accuracy_gru)
print(classification_report(y_test, y_pred_gru, target_names=['negative', 'neutral', 'positive']))

# Simpan model GRU
gru_model.save("gru_sentiment.h5")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 399s 315ms/step - accuracy: 0.8710 - loss: 0.3851 - val_accuracy: 0.9757 - val_loss: 0.0911
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 384s 307ms/step - accuracy: 0.9760 - loss: 0.0990 - val_accuracy: 0.9803 - val_loss: 0.0720
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 435s 302ms/step - accuracy: 0.9809 - loss: 0.0746 - val_accuracy: 0.9819 - val_loss: 0.0657
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 380s 301ms/step - accuracy: 0.9835 - loss: 0.0659 - val_accuracy: 0.9862 - val_loss: 0.0512
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 383s 302ms/step - accuracy: 0.9842 - loss: 0.0622 - val_accuracy: 0.9861 - val_loss: 0.0525
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 33s 26ms/step


GRU Accuracy: 0.986125
              precision    recall  f1-score   support

    negative       0.99      0.97      0.98     10071
     neutral       1.00      0.88      0.94      1946
    positive       0.98      1.00      0.99     27983

    accuracy                           0.99     40000
   macro avg       0.99      0.95      0.97     40000
weighted avg       0.99      0.99      0.99     40000



**penyimpanan model untuk inference**

In [9]:
# Simpan model CNN
cnn_model.save("cnn_sentiment.keras")

# Simpan model LSTM
lstm_model.save("lstm_sentiment.keras")

# Simpan model GRU
gru_model.save("gru_sentiment.keras")


# Simpan tokenizer agar bisa digunakan saat inference
import pickle
with open("tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
